# 7.4 Text Mining in Higher Ed – Topic Modeling — Code Brief

## Key Concepts

- **Topic modeling** is unsupervised — discovers recurring themes without pre-labeled data.
- **NMF** (Non-negative Matrix Factorization) — runs on TF-IDF vectors. Produces crisp, distinct topics; good for short survey comments.
- **LDA** (Latent Dirichlet Allocation) — runs on raw count vectors. Probabilistic; captures overlapping themes, better for longer documents.
- Both output: top-weighted words per topic (interpret via Gemini or domain knowledge) + a topic weight per response.
- `np.argmax` on the document-topic matrix assigns each student's **Dominant Topic** — turns weights into a usable categorical feature.

## Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random, os
import pandas as pd

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')
display(ML_Survey_Data)


## TF-IDF Vectors (Recap)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf_vec.fit_transform(ML_Survey_Data['Free_Response_Text'])
feature_names_tfidf = tfidf_vec.get_feature_names_out()

print("TF-IDF matrix:", tfidf_matrix.shape, "  (responses × terms)")
tfidf_matrix

tfidf_feature_names = tfidf_vec.get_feature_names_out()
df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)
df_tfidf_vectorized

## NMF Topic Modeling

In [ ]:
from sklearn.decomposition import NMF

N_TOPICS = 4  # start small; adjust after seeing results

nmf_model = NMF(n_components=N_TOPICS, random_state=42, max_iter = 500)
W_nmf = nmf_model.fit_transform(tfidf_matrix)   # document-topic
H_nmf = nmf_model.components_                   # topic-term

def print_top_words(H, feature_names, n_words=8, label='Topic'):
    for i, row in enumerate(H):
        top_idx = row.argsort()[:-n_words-1:-1]
        words = [feature_names[j] for j in top_idx]
        print(f"  {label} {i}: {', '.join(words)}")

print("NMF Topics (top 8 words each):")
print_top_words(H_nmf, feature_names_tfidf)


In [ ]:
# Prompt: Identify topics that summarize each of the sets of four words here: (Paste NMF Topic groups here)
topic_labels_nmf = {
    0: "Academic Experience & Expectations",
    1: "Study Habits & Peer Support",
    2: "College Experience & Personal Growth",
    3: "Academic Improvement & Challenges"
}
print("Summarized NMF topic labels:")
for t_id, label in topic_labels_nmf.items():
    print(f"Topic {t_id}: {label}")

## LDA Topic Modeling

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

count_vec = CountVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2), min_df=2)
count_matrix = count_vec.fit_transform(ML_Survey_Data['Free_Response_Text'])
feature_names_count = count_vec.get_feature_names_out()

lda_model = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=20)
W_lda = lda_model.fit_transform(count_matrix)

print("LDA Topics (top 8 words each):")
print_top_words(lda_model.components_, feature_names_count, label='Topic')


In [ ]:
topic_labels_lda = {
    0: "Performance & Feedback",
    1: "Engaging the Learning Process",
    2: "Study Habits & Support",
    3: "Learning Experiences and Goals"
}

print("LDA Topic Labels:")
for t_id, label in topic_labels_lda.items():
    print(f"Topic {t_id}: {label}")

## Assigning Dominant Topics

In [ ]:
import numpy as np
import plotly.express as px

# Add NMF and LDA results to our dataframe
df_Topic_Modeling = ML_Survey_Data[['SID','Free_Response_Text']]
df_Topic_Modeling['Dominant_Topic_NMF'] = np.argmax(W_nmf, axis=1)
df_Topic_Modeling['Dominant_Topic_LDA'] = np.argmax(W_lda, axis=1)

# Visualize NMF topic distribution
topic_counts = (df_Topic_Modeling['Dominant_Topic_NMF']
                .value_counts().sort_index().reset_index())
topic_counts.columns = ['Topic', 'Count']

fig = px.bar(topic_counts, x='Topic', y='Count',
             title='Dominant Topic Distribution — NMF',
             labels={'Topic': 'Topic Number', 'Count': 'Number of Responses'})
fig.show()

df_Topic_Modeling


## Reporting for IR Stakeholders

In [ ]:
def sample_comments(df, topic_col, topic_id, n=3, seed=42):
    subset = df[df[topic_col] == topic_id]
    return subset.sample(min(n, len(subset)), random_state=seed)['Free_Response_Text'].tolist()

total = len(df_Topic_Modeling)
print("=" * 60)
print("NMF TOPIC SUMMARY REPORT — Training Data")
print("=" * 60)
for t_id, label in topic_labels_nmf.items():
    count = (df_Topic_Modeling['Dominant_Topic_NMF'] == t_id).sum()
    pct = 100 * count / total
    examples = sample_comments(df_Topic_Modeling, 'Dominant_Topic_NMF', t_id)
    print(f"\nTopic {t_id}: {label}  ({count} responses, {pct:.1f}%)")
    for ex in examples:
        print(f"  • {ex}")


In [ ]:
def sample_comments(df, topic_col, topic_id, n=3, seed=42):
    subset = df[df[topic_col] == topic_id]
    return subset.sample(min(n, len(subset)), random_state=seed)['Free_Response_Text'].tolist()

total = len(df_Topic_Modeling)
print("=" * 60)
print("LDA TOPIC SUMMARY REPORT — Training Data")
print("=" * 60)
for t_id, label in topic_labels_lda.items():
    count = (df_Topic_Modeling['Dominant_Topic_LDA'] == t_id).sum()
    pct = 100 * count / total
    examples = sample_comments(df_Topic_Modeling, 'Dominant_Topic_LDA', t_id)
    print(f"\nTopic {t_id}: {label}  ({count} responses, {pct:.1f}%)")
    for ex in examples:
        print(f"  • {ex}")
